In [2]:
using JuMP
using Gurobi
using Random
using Dualization
using Plots
using DataFrames
using CSV
import XLSX
import JSON
using DataStructures


# Define parameters
tmin = 1
tmax = 10
T = [t for t in tmin:tmax]

unit = 1000

# Generate time periods
T = [t for t in tmin:tmax]

P = [
    "AJ_Vaccines",
    "BB_NCIPD",
    "China_National",
    "Bharat_Biotech",
    "Bilthoven",
    "Biological_E",
    "GSK",
    "Haffkine_Bio",
    "LG_Chem",
    "Merck_Sharp",
    "Panacea_Biotec",
    "PT_Bio",
    "Sanofi",
    "Serum_Institute",
    "Pfizer"
]

V = ["M", "MR", "MMR", "TT", "HepB", "Hib", "IPV", "OPV", "DT", "Td", "DTwP", "DTwP-Hib", "Penta", "Hexa", "HPV", "Rotavirus", "PCV"]

P_v = Dict("M" => ["Serum_Institute", "PT_Bio"], "MR" => ["Serum_Institute", "Biological_E"], "MMR" => ["Serum_Institute","GSK"],
"TT"=> ["Serum_Institute","PT_Bio","BB_NCIPD", "Biological_E"], "HepB" => ["Serum_Institute","LG_Chem"], "Hib" => ["Serum_Institute"],
"IPV" => ["LG_Chem","AJ_Vaccines","Bilthoven","Sanofi"],
"OPV" => ["Serum_Institute","PT_Bio","GSK","Sanofi","Panacea_Biotec","China_National","Bharat_Biotech","Haffkine_Bio"],
"DT" => ["PT_Bio","BB_NCIPD"], "Td" => ["Serum_Institute","PT_Bio","BB_NCIPD", "Biological_E"], "DTwP" => ["Serum_Institute","Biological_E"], "DTwP-Hib" => ["Serum_Institute"],
"Penta" => ["Serum_Institute","PT_Bio","Biological_E","LG_Chem","Panacea_Biotec"], "Hexa" => ["Sanofi"],
"HPV" => ["GSK","Merck_Sharp","China_National"], "Rotavirus" => ["Serum_Institute","GSK","Bharat_Biotech"], "PCV" => ["Serum_Institute","GSK","Pfizer"])

V_p = Dict()
for p in P
    vector_p = []
    for v in keys(P_v)
        if p in P_v[v]
            push!(vector_p, v)
        end
    end
    V_p[p] = vector_p
end


filename = "data/Vaccine_price_data.xlsx"    
vaccine_price_file = XLSX.readxlsx(filename)

r = Dict()
for v in V
    vaccine_price_raw = vaccine_price_file[string(v, " Pricing")]
    println(vaccine_price_raw)
    for row in 2:length(P_v[v])+1
        producer = vaccine_price_raw[row, 1]
        for col in 2:length(T)+1
            year = vaccine_price_raw[1, col]
            r[v, producer, year] = vaccine_price_raw[row, col]
        end
    end
end

25×11 XLSX.Worksheet: ["M Pricing"](A1:K25) 
3×11 XLSX.Worksheet: ["MR Pricing"](A1:K3) 
3×11 XLSX.Worksheet: ["MMR Pricing"](A1:K3) 
5×11 XLSX.Worksheet: ["TT Pricing"](A1:K5) 
35×11 XLSX.Worksheet: ["HepB Pricing"](A1:K35) 
2×11 XLSX.Worksheet: ["Hib Pricing"](A1:K2) 
18×11 XLSX.Worksheet: ["IPV Pricing"](A1:K18) 
28×11 XLSX.Worksheet: ["OPV Pricing"](A1:K28) 
3×11 XLSX.Worksheet: ["DT Pricing"](A1:K3) 
5×11 XLSX.Worksheet: ["Td Pricing"](A1:K5) 
3×11 XLSX.Worksheet: ["DTwP Pricing"](A1:K3) 
2×11 XLSX.Worksheet: ["DTwP-Hib Pricing"](A1:K2) 
34×11 XLSX.Worksheet: ["Penta Pricing"](A1:K34) 
2×11 XLSX.Worksheet: ["Hexa Pricing"](A1:K2) 
4×11 XLSX.Worksheet: ["HPV Pricing"](A1:K4) 
4×11 XLSX.Worksheet: ["Rotavirus Pricing"](A1:K4) 
4×11 XLSX.Worksheet: ["PCV Pricing"](A1:K4) 


In [7]:
r_avg = Dict()
for v in V
    for t in T
        total = 0.0
        for p in P_v[v]
            total += r[v, p, t]
        end
        average = total / length(P_v[v])
        r_avg[v, t] = average
    end
end

In [9]:
r_avg[("PCV", 5)]

2.747916666666667